In [1]:
# Getting imports and the model builder
import os
import sys
sys.path.insert(0, "../examples/open_deep_research")
sys.path.insert(0, "../baseline")

from smolagents import OpenAIModel
from dotenv import load_dotenv
load_dotenv()

MODEL_NAMES = ["Qwen/Qwen3.5-9B"]
WINDOW_SIZES = [1, 3, 5]


def build_model(model_name: str) -> OpenAIModel:
    if model_name in ["gpt-4o", "gpt-5.4-mini"]:
        return OpenAIModel(model_id=model_name, api_key=os.environ["OPENAI_API_KEY"])
    # Together requires enable_thinking nested inside chat_template_kwargs for vLLM-served
    # open-weight models -- verified this works for both Qwen3.7-Plus and Qwen3.5-9B.
    # client_kwargs timeout bounds a single API call so a stalled/hanging stream fails within
    # 5 minutes instead of hanging indefinitely -- evaluate_agent already catches such errors.
    return OpenAIModel(
        model_id=model_name,
        api_base="https://api.together.ai/v1/",
        api_key=os.environ["TOGETHER_API_KEY"],
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        client_kwargs={"timeout": 300.0},
    )

In [2]:
# Getting the tools setup and the MarkovReActCodeAgent builder
from smolagents.monitoring import LogLevel
from common_setup import assert_no_reasoning, build_tools
from smolagents.markov_react import MarkovReActCodeAgent


def build_agent(model, model_name: str, window_size: int, tools):
    return MarkovReActCodeAgent(
        tools=tools,
        model=model,
        max_steps=50,
        verbosity_level=LogLevel.DEBUG,
        additional_authorized_imports=["pandas", "numpy", "PIL", "json", "io", "zipfile", "csv", "openpyxl"],
        window_size=window_size,
        stream_outputs="Qwen" in model_name,
        step_callbacks=[assert_no_reasoning] if "Qwen" in model_name else None,
    )

/Users/poorvag/Work/smolagents/.venv/lib/python3.14/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


## Fermi (RealFP) dataset

Same RealFP split, `fermi_scorer`, and `FERMI_ANSWER_INSTRUCTION` used in
[naiveReAct_fermi.ipynb](../baseline/naiveReAct_fermi.ipynb), reused here so MarkovReAct stays
directly comparable to naive ReAct on Fermi -- same dataset, same scorer, same prompt suffix.

In [3]:
# Load the Fermi RealFP validation split
import pandas as pd
from common_setup import evaluate_agent
from fermi_setup import load_fermi_dataset, fermi_scorer, FERMI_ANSWER_INSTRUCTION

eval_ds = load_fermi_dataset(split="val")
print(f"Loaded {len(eval_ds)} examples")

Loaded 125 examples


In [4]:
# One model (and its tools) is built once, then reused across all 3 window sizes.
all_results = {}

for model_name in MODEL_NAMES:
    model = build_model(model_name)
    tools, ti_tool, visualizer = build_tools(model)

    for window_size in WINDOW_SIZES:
        agent = build_agent(model, model_name, window_size, tools)
        safe_model_name = model_name.replace("/", "_")
        pickle_dir = f"markov_fermi_{safe_model_name}_w{window_size}"
        print(f"\n=== {model_name}, window_size={window_size} ===")
        results = evaluate_agent(
            agent,
            eval_ds,
            ti_tool,
            visualizer,
            output_file=f"{pickle_dir}.jsonl",
            pickle_dir=pickle_dir,
            scorer=fermi_scorer,
            question_suffix=FERMI_ANSWER_INSTRUCTION,
        )
        all_results[(model_name, window_size)] = results


=== Qwen/Qwen3.5-9B, window_size=1 ===

[1/125] (cached) If all but 1 million people on Earth died, how far (on average) would you have to walk to meet someo...
  ✓ | cached

[2/125] (cached) How much water would need to be evaporated to turn Arizona tropical?...
  ✗ | cached

[3/125] (cached) If a gallon of paint is used to coat 400 ft2 of walls, how thick, cm, is the paint film?...
  ✓ | cached

[4/125] (cached) How much space would be required to fit 1 individual of each species of plants and animals?...
  ✗ | cached

[5/125] (cached) How many canaries in the world are currently flying?...
  ✓ | cached

[6/125] (cached) How old are you if you are a million hours old?...
  ✓ | cached

[7/125] (cached) How many mice could pull the Maus tank?...
  ✓ | cached

[8/125] (cached) How many times does your heart beat per week?...
  ✓ | cached

[9/125] (cached) What do you estimate the total weight of all the Covid-19 viruses in the world?...
  ✓ | cached

[10/125] (cached) How many people a

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ If human skin could photosynthesise with the same efficiency as a leaf, would it produce enough energy for us   │
│ to survive or would we still need to eat food?                                                                  │
│                                                                                                                 │
│ Give your final answer as a number with units matching the question (e.g. "22.3 km", "79 g"; use a bare number  │
│ like "33000" only if the question has no natural unit). Do not include filler words like "about" or "roughly",  │
│ or thousands separators. Write the full number of digits (e.g. "256000000000") rather than word multipliers     │
│ like "million" or "billion".                                                                                    │
│                                                                                                                 │
╰─ OpenAIModel - Qwen/Qwen3.5-9B ─────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output()

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  results = web_search("human daily energy requirement joules")                                                    
  print("Human energy search results:")                                                                            
  print(results)                                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

...

Output()

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(18750)                                                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 18750

[Step 29: Duration 6.28 seconds| Input tokens: 283,083 | Output tokens: 6,360]

  ✓ | 116.1s | 29 steps | tokens: TokenUsage(input_tokens=283083, output_tokens=6360, total_tokens=289443)

[28/125] (cached) How many Gluesticks Fit in a Horse?...
  ✓ | cached

[29/125] (cached) What is the total carcass weight of roadkill in USA each year?...
  ✗ | cached

[30/125] (cached) How many kilometers would the Earth have travelled during the time elapsed between two full moons?...
  ✓ | cached

[31/125] (cached) How many nerd ropes would it take to go around the Earth?...
  ✓ | cached

[32/125] (cached) How many golf balls could you fit in a 5 gallon water cooler jug?...
  ✓ | cached

[33/125] (cached) How many cells are needed to visible to a person?...
  ✓ | cached

[34/125] (cached) How much does it cost to leave a light on for an entire year?...
  ✓ | cached

[35/125] (cached) How many model rockets would it take to propel a 140 lb human?...
  ✓ | cached

[36/125] (cached) The Ontario government decides that winter is too depressing and that what is really needed is to

In [5]:
rows = []
for (model_name, window_size), results in all_results.items():
    df = pd.DataFrame(results)
    total = len(df)
    avg_score = df["is_correct"].mean()  # continuous 0-1 order-of-magnitude accuracy metric, not a boolean fraction
    total_tokens = df["token_counts"].apply(lambda x: x.get("total_tokens", 0))
    rows.append({
        "Model": model_name,
        "window_size": window_size,
        "Avg Score": round(avg_score, 3),
        "Avg Steps": round(df["num_steps"].mean(), 1),
        "Avg Tokens": round(total_tokens.mean()),
    })

pd.DataFrame(rows)

,Model,window_size,Avg Score,Avg Steps,Avg Tokens
0,Qwen/Qwen3.5-9B,1,0.546,21.0,88980
1,Qwen/Qwen3.5-9B,3,0.525,16.8,109017
2,Qwen/Qwen3.5-9B,5,0.585,14.3,121515
